# Validación offline del modelo

Este notebook se ejecuta desde la carpeta del modelo y no usa Internet.

In [1]:
import os

os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ['HF_DATASETS_OFFLINE'] = '1'

In [2]:
from pathlib import Path
import json

MODEL_PATH = Path.cwd().resolve()
metadata_path = MODEL_PATH / 'model-metadata.json'
metadata = json.loads(metadata_path.read_text(encoding='utf-8'))
required_fields = {'name', 'model_type', 'source', 'revision', 'framework', 'python_target'}
missing_fields = required_fields - metadata.keys()
assert not missing_fields, f'Metadata incompleta: {sorted(missing_fields)}'
assert metadata['name'] == MODEL_PATH.name, 'El nombre no coincide con la carpeta'
assert metadata['model_type'] == 'embedding', 'Tipo de modelo incorrecto'
print(f'Model path: {MODEL_PATH.resolve()}')
for field in ('name', 'model_type', 'source', 'revision', 'framework', 'python_target'):
    print(f'{field}: {metadata[field]}')

Model path: <local model directory>
name: paraphrase-multilingual-MiniLM-L12-v2
model_type: embedding
source: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
revision: e8f8c211226b894fcb81acc59f3b34ba3efd5f42
framework: sentence-transformers
python_target: 3.11


In [3]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    str(MODEL_PATH),
    local_files_only=True,
    trust_remote_code=False,
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [4]:
import numpy as np

texts = [
    'Cliente solicita financiamiento para capital de trabajo.',
    'La empresa presenta crecimiento sostenido de ventas.',
    'La compañía mantiene una posición financiera estable.',
]
embeddings = model.encode(
    texts,
    normalize_embeddings=True,
    show_progress_bar=False,
)
assert embeddings.shape[0] == len(texts)
assert embeddings.ndim == 2
assert embeddings.shape[1] > 0
assert np.isfinite(embeddings).all()
print(f'modelo: {metadata["name"]}')
print(f'cantidad de textos: {len(texts)}')
print(f'shape: {embeddings.shape}')
print(f'dimensión del embedding: {embeddings.shape[1]}')
print(f'dtype: {embeddings.dtype}')
print('resultado: OK')

modelo: paraphrase-multilingual-MiniLM-L12-v2
cantidad de textos: 3
shape: (3, 384)
dimensión del embedding: 384
dtype: float32
resultado: OK


In [5]:
print('VALIDATION OK')
print('Modelo cargado y ejecutado correctamente en modo offline.')

VALIDATION OK
Modelo cargado y ejecutado correctamente en modo offline.
